# 1 - Explore the datasets

We start from three publicly available datasets, in the formats in which they are distributed:

| Dataset                      | Version/Year | Resolution |  Provider                   | Original format             |
|------------------------------|--------------|------------|-----------------------------|-----------------------------|
| AHN Digital Surface Model    | 4            | 0.5 m      | [PDOK / AHN][AHN]           | GeoTIFF tiles (1 km x 1 km) |
| Aerial images NL (RGB)       | 2024         | 8 cm       | [PDOK / Beeldmateriaal][BM] | GeoTIFF tiles (1 km x 1 km) |
| Daily mean temperature (Tg1) | 5            | 1 km       | [KNMI][Tg1]                 | NetCDF3, one file per day   |

[AHN]: https://www.ahn.nl/
[BM]: https://www.beeldmateriaal.nl/
[Tg1]: https://dataplatform.knmi.nl/dataset/tg1-5 

A copy of a small subset of them is on the object store. Which properties make these files more or less suitable for cloud-native access?

In [ ]:
import os

import fsspec

from utils import ENDPOINT_URL, BUCKET_URI, print_raster_info, print_netcdf_info

In [ ]:
fs = fsspec.filesystem("s3", anon=True, endpoint_url=ENDPOINT_URL)   

In [ ]:
print(fs.tree(BUCKET_URI, recursion_limit=1))

## AHN Digital Surface Model

In [ ]:
print(fs.tree(f"{BUCKET_URI}/AHN-4", recursion_limit=2, max_display=5))

In [ ]:
dsm_path, *_ = fs.glob(f"{BUCKET_URI}/AHN-4/DSM05/*.TIF")
print_raster_info(dsm_path)

The GeoTIFF is already internally tiled (256 x 256 blocks) and compressed (DEFLATE), but it has **no overviews** and GDAL doesn't recognise it as a COG (`Layout: None`).

## RGB Aerial Images

In [ ]:
print(fs.tree(f"{BUCKET_URI}/aerial-images-NL-2024", recursion_limit=2, max_display=5))

In [ ]:
rgb_path, *_ = fs.glob(f"{BUCKET_URI}/aerial-images-NL-2024/RGB/*.tif")
print_raster_info(rgb_path)

Tiled and JPEG-compressed, but 12500 x 12500 pixels without overviews: an overview-level image of a tile requires reading the entire file!

## Gridded Daily Mean Temperature

In [ ]:
print(fs.tree(f"{BUCKET_URI}/KNMI-Tg1-5", recursion_limit=2, max_display=5))

In [ ]:
nc_path, *_ = fs.glob(f"{BUCKET_URI}/KNMI-Tg1-5/*/*.nc")
print_netcdf_info(nc_path)

NetCDF3: no chunking, no compression. One small file per day: fine to get a map for a given day, but extracting the time series at one location means touching 365 files.

> ## Discussion
> For each dataset, which access pattern you think users will need most? What you would change in the file layout to support it?